In [ ]:
FIGURE = 'SSMIrz'
from pathlib import Path
from contextlib import contextmanager
import os
import sys
import gc
import mmap
import json
import hashlib
import shutil
import numpy as np
import pandas as pd
import netCDF4 as nc
_NETCDF_DATASET = nc.Dataset
HERE = Path.cwd().resolve()
if not (HERE / 'Inputs').is_dir():
    for candidate in [HERE / 'Figure' / FIGURE, HERE / FIGURE, HERE / 'Data Processing' if FIGURE in ['SSMIrz','Rmax_Ts'] else HERE]:
        if (candidate / 'Inputs').is_dir():
            HERE = candidate
            break
INPUT_DIR = HERE / 'Inputs'
OUTPUT_DIR = HERE / 'Outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
ROOT = HERE
DATA = INPUT_DIR
OUTPUT = OUTPUT_DIR
__file__ = str(HERE / (FIGURE + '.ipynb'))
sys.argv = ['notebook']

def open_nc(path, mode='r', **kwargs):
    if 'memory' in kwargs:
        return _NETCDF_DATASET(path, mode, **kwargs)
    path = Path(path).resolve()
    previous = Path.cwd()
    try:
        os.chdir(path.parent)
        return _NETCDF_DATASET(path.name, mode, **kwargs)
    finally:
        os.chdir(previous)

def asset_path(key):
    if key in ['asset_e2603c619a67', 'asset_5e3c32038bfb']:
        return INPUT_DIR / 'study_mask.tif'
    if key == 'asset_beb53c9b3525':
        return INPUT_DIR / 'SPEI_k01-k24.nc'
    if key == 'asset_174bc324d489':
        return INPUT_DIR / 'SSMIrz_k01-k24.nc'
    return OUTPUT_DIR

def output_path(name=''):
    path = OUTPUT_DIR / name
    path.mkdir(parents=True, exist_ok=True)
    return path

def module_path(name=''):
    return str(HERE)

def font_path(filename):
    from matplotlib.font_manager import FontProperties, findfont
    family = 'Times New Roman' if filename.lower().startswith('times') else 'Courier New' if filename.lower().startswith('cour') else 'Arial'
    return findfont(FontProperties(family=family))

def array(value):
    return np.asarray(np.ma.filled(value.astype(float), np.nan))

@contextmanager
def mapped(path):
    with Path(path).open('rb') as stream:
        mm = mmap.mmap(stream.fileno(), 0, access=mmap.ACCESS_READ)
        dataset = open_nc('input.nc', memory=mm)
        try:
            yield dataset
        finally:
            dataset.close()
            gc.collect()
            mm.close()


In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import norm
in_dir = INPUT_DIR / 'SMrz_monthly'
out_file = OUTPUT / 'SSMIrz_2001_2020_k01-k24.nc'
years = range(1997, 2021)
scales = range(1, 25)
files = []
for_years = []
for y in years:
    f = os.path.join(in_dir, f'SMrz_{y}_GLEAM_v4.3a_MO_0.5deg.nc')
    if os.path.exists(f):
        files.append(f)
        for_years.append(y)
    else:
        print(f'Warning: missing file: {f}')
print(f'Found {len(files)} files:')
print(files[:3], '...', files[-3:])
ds = xr.open_mfdataset(files, combine='nested', concat_dim='time', decode_times=False, parallel=False, engine='h5netcdf')
print(ds)
candidate_vars = list(ds.data_vars)
print('Data variables:', candidate_vars)
varname = 'SMrz'
if varname not in candidate_vars:
    raise ValueError(f"variable_name '{varname}' missing_from_dataset, available_variables: {candidate_vars}")
sm = ds[varname]
expected_time = pd.date_range('1997-01-01', '2020-12-01', freq='MS')
if sm.sizes['time'] != len(expected_time):
    raise ValueError(f"Time length mismatch: data has {sm.sizes['time']}, expected {len(expected_time)}")
sm = sm.assign_coords(time=expected_time)
print('Loading data into memory, this may take a moment...')
sm = sm.load()
print('Load complete.')

def empirical_to_normal_vectorized(x):
    x = np.asarray(x, dtype=np.float64)
    n = x.shape[-1]
    valid_mask = np.isfinite(x)
    all_valid = valid_mask.all(axis=-1)
    z = np.full(x.shape, np.nan, dtype=np.float32)
    if not np.any(all_valid):
        return z
    order = np.argsort(x, axis=-1)
    ranks = np.empty_like(order, dtype=np.float64)
    idx = np.arange(1, n + 1)
    np.put_along_axis(ranks, order, idx, axis=-1)
    p = (ranks - 0.5) / n
    zz = norm.ppf(p)
    z = np.where(all_valid[..., None], zz, np.nan).astype(np.float32)
    return z
ssmi_list = []
for k in scales:
    print(f'Processing scale k={k:02d}')
    sm_k = sm.rolling(time=k, min_periods=k).mean()
    sm_k = sm_k.sel(time=slice('2001-01-01', '2020-12-01'))
    monthly_z_list = []
    for m in range(1, 13):
        sm_k_m = sm_k.where(sm_k['time.month'] == m, drop=True)
        z_m = xr.apply_ufunc(empirical_to_normal_vectorized, sm_k_m, input_core_dims=[['time']], output_core_dims=[['time']], vectorize=False, output_dtypes=[np.float32])
        monthly_z_list.append(z_m)
    z_k = xr.concat(monthly_z_list, dim='time').sortby('time')
    z_k = z_k.expand_dims(scale=[k])
    z_k.name = 'SSMIrz'
    ssmi_list.append(z_k)
ssmi = xr.concat(ssmi_list, dim='scale')
ssmi.name = 'SSMIrz'
ssmi.attrs['long_name'] = 'Multi-timescale standardized root-zone soil moisture index'
ssmi.attrs['description'] = 'SSMIrz derived from GLEAM root-zone soil moisture. Antecedent mean SMrz was calculated using moving windows of 1-24 months, then transformed to standard normal variates using calendar-month-, grid-cell-, and timescale-specific empirical distributions.'
ssmi.attrs['units'] = '1'
ssmi.attrs['negative_values'] = 'drier-than-normal root-zone moisture conditions'
ssmi.attrs['positive_values'] = 'wetter-than-normal root-zone moisture conditions'
out_ds = ssmi.to_dataset()
out_ds.attrs['source_data'] = 'GLEAM v4.3a monthly root-zone soil moisture'
out_ds.attrs['input_period'] = '1997-2020'
out_ds.attrs['analysis_period'] = '2001-2020'
out_ds.attrs['timescales'] = '1-24 months'
out_ds.attrs['method'] = 'Rolling mean of SMrz followed by empirical probability normalization to standard normal distribution for each grid cell, calendar month, and timescale.'
os.makedirs(os.path.dirname(out_file), exist_ok=True)
encoding = {'SSMIrz': {'zlib': True, 'complevel': 4, 'dtype': 'float32', '_FillValue': np.float32(np.nan)}}
out_ds.to_netcdf(out_file, encoding=encoding, engine='h5netcdf')
print('Done!')
print(out_file)